In [ ]:
# Cell 1 — Load trained model
"""
03_inference.ipynb
==================
Interactive predictions with a trained TGNN-Solv model.

This notebook is the exploratory path. For scripted reports, use
`scripts/evaluation/evaluate_complete.py`, `scripts/evaluation/error_analysis.py`,
`scripts/evaluation/validate_physics.py`, and
`scripts/experiments/generate_paper_figures.py`.

For richer uncertainty analysis, see `04_evaluation.ipynb`.
This notebook now also includes a lightweight applicability-domain spot check.
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
from tgnn_solv.inference import (
    load_model,
    predict_solubility,
    temperature_scan,
    interpret_prediction,
)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
MODEL_PATH = CHECKPOINT_DIR / "tgnn_solv_trained.pt"
model, cfg = load_model(str(MODEL_PATH), DEVICE)
print(f"Loaded {MODEL_PATH} on {DEVICE}")


## Formulas for single-point inference and temperature scan

At inference time, the main object is the decomposition of the predicted
solubility into a crystal contribution and a non-ideal contribution:

$$
\ln x_2^{\mathrm{phys}} = -\Phi - \ln \gamma_2.
$$

The final answer after bounded correction is

$$
\ln x_2^{\mathrm{final}} = \ln x_2^{\mathrm{phys}} + (1-c)\,r,
$$

where \(c\) is the correction-gate confidence and \(r\) is the bounded residual
in the space of solver-consistent solutions. The notebook API then also returns

$$
x_2 = \exp\!\left(\ln x_2^{\mathrm{final}}\right),
\qquad
\gamma_2 = \exp\!\left(\ln \gamma_2\right).
$$

For `temperature_scan`, a discrete temperature grid is constructed:

$$
T_j = T_{\min} + \frac{j}{n-1}\left(T_{\max} - T_{\min}\right),
\qquad j = 0,\dots,n-1,
$$

and the same inference pipeline is run at every grid point. In other words,
the temperature scan is not a separate surrogate model; it is repeated calls
to `predict_solubility(...)` over a temperature grid.


## Step 1. Inspect one example end-to-end

It is useful to start with a single system and inspect not only `ln(x_2)`,
but also the decomposition, `T_m`, `\tau_{12}`, Hansen parameters, and the
correction magnitude. This gives a quick sense of whether the prediction looks
physically plausible.


In [ ]:
# Cell 2 — Single prediction with full report
result = predict_solubility(
    model,
    solute_smiles="CC(=O)Nc1ccc(O)cc1",   # paracetamol
    solvent_smiles="CCO",                   # ethanol
    T=298.15,
)
print(interpret_prediction(result))

## Step 2. Compare one solute across a solvent panel

The next step is qualitative screening: hold the solute fixed and inspect how
the model ranks a set of solvents. This is often the easiest place to spot
errors in the non-ideal term and in the chemical sensitivity of the pair representation.


In [ ]:
# Cell 3 — Compare multiple solvents
solvents = {
    "Water": "O",
    "Ethanol": "CCO",
    "Methanol": "CO",
    "Acetone": "CC(=O)C",
    "Hexane": "CCCCCC",
    "Toluene": "Cc1ccccc1",
    "DMSO": "CS(=O)C",
    "Chloroform": "ClC(Cl)Cl",
}

solute = "CC(=O)Nc1ccc(O)cc1"  # paracetamol
print(f"{'Solvent':15s} {'x₂':>10s} {'ln(x₂)':>8s} {'γ₂':>8s} {'Ra':>6s}")
print("-" * 50)

for name, smi in solvents.items():
    r = predict_solubility(model, solute, smi, T=298.15)
    print(f"{name:15s} {r['x2']:10.5f} {r['ln_x2']:8.3f} "
          f"{r['gamma_2']:8.2f} {r['Ra']:6.1f}")

## Step 3. A temperature curve for one pair

This block checks not only the solubility level, but also the shape of the
temperature dependence. Looking at `x_2(T)`, `ln(x_2)` in `1/T` coordinates,
and `\gamma_2(T)` together helps identify what is actually driving the curve.


In [ ]:
# Cell 4 — Temperature scan with plot
import matplotlib.pyplot as plt

scan = temperature_scan(
    model,
    solute_smiles="CC(=O)Nc1ccc(O)cc1",
    solvent_smiles="CCO",
    T_min=270, T_max=340, n_points=30,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Solubility vs T
axes[0].plot(scan["T"] - 273.15, scan["x2"], "o-", ms=4)
axes[0].set_xlabel("Temperature (°C)")
axes[0].set_ylabel("x₂ (mole fraction)")
axes[0].set_title("Paracetamol / Ethanol")

# ln(x2) vs 1/T (van 't Hoff style)
axes[1].plot(1000 / scan["T"], scan["ln_x2"], "s-", ms=4, color="green")
axes[1].set_xlabel("1000/T (K⁻¹)")
axes[1].set_ylabel("ln(x₂)")
axes[1].set_title("van 't Hoff plot")

# γ₂ vs T
axes[2].plot(scan["T"] - 273.15, scan["gamma_2"], "^-", ms=4, color="red")
axes[2].set_xlabel("Temperature (°C)")
axes[2].set_ylabel("γ₂")
axes[2].set_title("Activity coefficient")

plt.tight_layout()
plt.show()

# Monotonicity check
#diffs = scan["ln_x2"].diff().dropna()
#print(f"Monotonically increasing: {(diffs >= -0.01).all()}")

## Step 4. Batch inference for a small set of systems

After individual examples, it is convenient to move to a small batch and compare
systems in table form. This is a useful intermediate format between manual case-by-case
interpretation and a full benchmark script.


In [ ]:
# Cell 5 — Batch prediction
import pandas as pd

systems = [
    ("CC(=O)Nc1ccc(O)cc1", "O", 298.15),       # paracetamol / water
    ("CC(=O)Nc1ccc(O)cc1", "CCO", 298.15),      # paracetamol / ethanol
    ("c1ccc2ccccc2c1", "c1ccccc1", 298.15),      # naphthalene / benzene
    ("c1ccc2ccccc2c1", "O", 298.15),             # naphthalene / water
    ("OC(=O)c1ccccc1", "CCO", 298.15),           # benzoic acid / ethanol
    ("OC(=O)c1ccccc1", "O", 298.15),             # benzoic acid / water
    ("CC(=O)Oc1ccccc1C(=O)O", "CCO", 298.15),   # aspirin / ethanol
    ("CC(=O)Oc1ccccc1C(=O)O", "O", 298.15),     # aspirin / water
]

results = []
for sol, slv, T in systems:
    r = predict_solubility(model, sol, slv, T)
    results.append(r)

df = pd.DataFrame(results)
display_cols = ["solute", "solvent", "x2", "ln_x2", "gamma_2",
                "T_m", "dH_fus", "Ra", "correction"]
print(df[display_cols].to_string(index=False, float_format="{:.4f}".format))

## Applicability-domain formulas

Inference-time OOD screening in the current code combines two parts.
For the latent pair vector \(z\), a Mahalanobis distance is computed:

$$
d_M(z) = \sqrt{(z - \mu)^\top \Sigma^{-1} (z - \mu)},
$$

where \(\mu\) and \(\Sigma\) are estimated from the training loader after fitting.

For solute and solvent separately, the score also uses the maximum Tanimoto similarity
to the training fingerprints:

$$
s_{\mathrm{sol}} = \max_{u \in \mathcal{T}_{\mathrm{sol}}}
\operatorname{Tan}(q_{\mathrm{sol}}, u),
\qquad
s_{\mathrm{slv}} = \max_{v \in \mathcal{T}_{\mathrm{slv}}}
\operatorname{Tan}(q_{\mathrm{slv}}, v).
$$

The current verdict is a logical conjunction:

$$
\mathbb{1}_{\mathrm{in\_domain}} =
\mathbb{1}\left[d_M \le d_{\mathrm{cut}}\right]
\land
\mathbb{1}\left[s_{\mathrm{sol}} \ge s_{\min}\right]
\land
\mathbb{1}\left[s_{\mathrm{slv}} \ge s_{\min}\right].
$$

The returned `confidence` aggregates both components:

$$
c_{\mathrm{AD}} = \frac12
\max\!\left(0, 1 - \frac{d_M}{2 d_{\mathrm{cut}}}\right)
+ \frac12 \min\!\left(s_{\mathrm{sol}}, s_{\mathrm{slv}}\right).
$$

Important: in the current implementation, leverage does not participate in either the verdict or the final confidence score.


## Step 5. Check in-domain vs OOD directly at inference time

`predict_solubility(...)` does not by itself block OOD queries, so a separate
AD check is needed whenever you want to make a trust decision. This is especially
important for rare solvents, unusual scaffolds, and temperature regimes outside the
usual training distribution.


In [ ]:
# Cell 6 — Applicability Domain spot check
import pandas as pd
from tgnn_solv.data import PROCESSED_DIR, make_loaders
from tgnn_solv.domain import ApplicabilityDomain

# AD is not baked into `predict_solubility`; fit it once on the training split
# and call it alongside inference when you want an in-domain / OOD screen.
# Current AD = pair-latent Mahalanobis + nearest-neighbor Tanimoto.
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")
train_loader, _, _ = make_loaders(train_df, val_df, test_df, batch_size=cfg.batch_size)

ad = ApplicabilityDomain(model, train_loader)
print(ad.report(
    solute_smiles="CC(=O)Nc1ccc(O)cc1",
    solvent_smiles="CCO",
    T=298.15,
))

## Step 6. Compare decomposition terms across several systems

The final chart aggregates the decomposition over multiple examples. This is already
closer to research use than to simple inference: it shows where the crystal term dominates,
where non-ideality dominates, and where the correction branch is compensating for weaknesses
in the physics path.


In [ ]:
# Cell 7 — Interpretability: decomposition chart
import matplotlib.pyplot as plt
import numpy as np

# Use results from Cell 5
names = [f"{r['solute'][:20]}\n{r['solvent'][:10]}" for r in results]
phi = [-r["Phi"] for r in results]
lng = [-r["ln_gamma_2"] for r in results]
corr = [r["correction"] for r in results]

x_pos = np.arange(len(results))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x_pos - width, phi, width, label="-Φ (crystal)", color="steelblue")
ax.bar(x_pos, lng, width, label="-ln(γ_2) (interaction)", color="coral")
ax.bar(x_pos + width, corr, width, label="correction", color="gray")

ax.set_xticks(x_pos)
ax.set_xticklabels(names, fontsize=7, rotation=45, ha="right")
ax.set_ylabel("Contribution to ln(x_2)")
ax.set_title("Solubility decomposition: crystal + interaction + correction")
ax.legend()
ax.axhline(0, color="black", lw=0.5)
plt.tight_layout()
plt.show()